In [151]:
import random
import time

#Basisklasse für Charaktere im Spiel

class Charakter:
    def __init__(self, name, leben, schaden,ausweicchen=0.0, ruestung=0.0):
        self.name = name
        self.leben  = leben
        self.schaden = schaden
        self.ausweicchen = ausweicchen
        self.ruestung = ruestung

    def ist_am_leben(self):
        return self.leben > 0   

    def trifft(self, ziel):
        if random.random() <= ziel.ausweicchen:
            return False
        return True
    
    def schadenswurf(self, basis_schaden):
        faktor = random.uniform(0.8, 1.2)
        return max(1, int(basis_schaden * faktor))
    
    def schaden_anwenden(self, ziel, roher_schaden):
        final = max(0, roher_schaden - ziel.ruestung)
        ziel.leben -= final
        return final
    

    def angreifen(self, ziel):

        print(f"{self.name} greift {ziel.name} an!")

        if not self.trifft(ziel):
            print(f"{ziel.name} weicht dem Angriff von {self.name} aus!")
            return
        
        roher = self.schadenswurf(self.schaden)
        final = self.schaden_anwenden(ziel, roher)
        print(f"{self.name} trifft {ziel.name} für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")

        

        
#Klasse für den Spielercharakter
class Krieger(Charakter):
    def __init__(self, name):
        super().__init__(name, leben=100, schaden=14, ausweicchen=0.05, ruestung=2)
        self.stamina = 30

    def angreifen(self, ziel):
        heavy = self.stamina >= 10 and random.random() < 0.2
        if heavy:
            self.stamina -= 10
            print(f"{self.name} führt einen starken Angriff durch! und hat noch {self.stamina} Stamina übrig.")
            if not self.trifft(ziel):
                print(f"{ziel.name} weicht dem starken Angriff von {self.name} aus!")
                return
            roher = self.schadenswurf(int(self.schaden * 1.5))
            final = self.schaden_anwenden(ziel, roher)
            print(f"{self.name} trifft {ziel.name} mit einem starken Angriff für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
        else:
            self.stamina = min(30, self.stamina + 3)  # Stamina regeneriert sich langsam    
            super().angreifen(ziel)

class Bogenschütze(Charakter):
    def __init__(self, name):
        super().__init__(name, leben=100, schaden=10, ausweicchen=0.1, ruestung=1)
        self.pfeile = 6

    def angreifen(self, ziel):

        if self.pfeile <= 0:
            print(f"{self.name} hat keine Pfeile mehr und muss mit der Faust angreifen!")
            super().angreifen(ziel)
            return
        
        if not self.trifft(ziel):
            print(f"{self.name} wirft einen Pfeil, aber {ziel.name} weicht ihm aus!")
            self.pfeile -= 1
            print(f"{self.name} hat noch {self.pfeile} Pfeile übrig.")
            return

        basis = self.schaden * random.uniform(1.0, 2.2)
        roher = self.schadenswurf(basis)
        final = self.schaden_anwenden(ziel, roher)
        print(f"{self.name} trifft {ziel.name} mit einem Pfeil für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
        self.pfeile -= 1
        print(f"{self.name} hat noch {self.pfeile} Pfeile übrig.")


        
class Magier(Charakter):
    def __init__(self, name):
        super().__init__(name, leben=92, schaden=12, ausweicchen=0.10, ruestung=0)
        self.mana = 50

    def angreifen(self, ziel):
        self.mana = min(50, self.mana + 5)  # Mana regeneriert sich langsam
        if self.mana >= 14 and random.random() < 0.35:
            self.mana -= 14
            print(f"{self.name} wirkt einen mächtigen Zauber! und hat noch {self.mana} Mana übrig.")
            if random.random() < 0.2:
                print(f"{self.name} hat den Zauber nicht richtig gezaubert und verliert 10 Mana!")
                self.mana -= 10
                return
            roher = self.schadenswurf(int(self.schaden * 2.5))
            final = self.schaden_anwenden(ziel, roher)
            print(f"{self.name} trifft {ziel.name} mit einem Zauber für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
            return
        

        if self.mana >= 8 and random.random() < 0.55:
            self.mana -= 8
            print(f"{self.name} wirkt einen Feuerball! und hat noch {self.mana} Mana übrig.")
            if not self.trifft(ziel):
                print(f"{ziel.name} weicht dem Feuerball von {self.name} aus!")
                return
            roher = self.schadenswurf(int(self.schaden * 1.8))
            final = self.schaden_anwenden(ziel, roher)
            print(f"{self.name} trifft {ziel.name}  für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
            return
        super().angreifen(ziel)
            
        

class Heiler(Charakter):
    def __init__(self, name):
        super().__init__(name, leben=95, schaden=10, ausweicchen=0.12, ruestung=1)
        self.heilung = 20

    def angreifen(self, ziel):
        if self.ist_am_leben() and self.heilung > 0 and self.leben < 50:
            if self.leben < 20:
                self.leben += self.heilung
                self.heilung = 0 
                roher = self.schadenswurf(self.schaden * 2.5)
                final = self.schaden_anwenden(ziel, roher)
                print(f"{self.name} setzt einen verzweifelten Heilzauber ein und heilt sich selbst! {self.name} hat keine Heilungskraft mehr übrig.")
                print(f"(Roh: {roher}, Rüstung: {ziel.ruestung})")
                return
            if random.random() < 0.5:
                self.leben += self.heilung
                self.leben = min(self.leben, 95)  # Maximalleben begrenzen
                self.heilung = max(5, self.heilung - 5)  # Heilung wird schwächer
                print(f"{self.name} heilt sich selbst um {self.heilung + 5} Leben! und hat noch {self.heilung} Heilungskraft übrig.")
                super().angreifen(ziel) 
                return
        super().angreifen(ziel)

#Klasse für Gegner

class Gegner(Charakter):
    def __init__(self, name, leben, schaden, ausweicchen=0.0, ruestung=0.0):
        super().__init__(name, leben, schaden, ausweicchen, ruestung)   


class Goblin(Gegner):
    def __init__(self):
        super().__init__("Goblin", leben=75, schaden=8, ausweicchen=0.4, ruestung=0)

    def angreifen(self, ziel):
        if random.random() < 0.2:
            print(f"{self.name} führt einen hinterhältigen Angriff durch!")
            roher = self.schadenswurf(int(self.schaden * 2.5))
            final = self.schaden_anwenden(ziel, roher)
            print(f"{self.name} trifft {ziel.name} mit einem hinterhältigen Angriff für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
            return
        
        super().angreifen(ziel)

class Ork(Gegner):
    def __init__(self):
        super().__init__("Ork", leben=90, schaden=14, ausweicchen=0.17, ruestung=1)
    def angreifen(self, ziel):
        if random.random() < 0.3:
            print(f"{self.name} führt einen wütenden Angriff durch!")
            roher = self.schadenswurf(int(self.schaden * 1.5))
            final = self.schaden_anwenden(ziel, roher)
            print(f"{self.name} trifft {ziel.name} mit einem wütenden Angriff für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
            return
        super().angreifen(ziel)

class Drache(Gegner):
    def __init__(self):
        super().__init__("Drache", leben=100, schaden=16, ausweicchen=0.1, ruestung=2)
    def angreifen(self, ziel):
        if random.random() < 0.25:
            print(f"{self.name} speit Feuer!")
            roher = self.schadenswurf(int(self.schaden * 1.3))
            final = self.schaden_anwenden(ziel, roher)
            print(f"{self.name} trifft {ziel.name} mit einem Feuerstoß für {final} Schaden! (Roh: {roher}, Rüstung: {ziel.ruestung})")
            return
        super().angreifen(ziel)



#Arena

def arena(held, gegner):
    print(f"Willkommen in der Arena, {held.name}!")
    print(f"Dein Gegner ist {gegner.name}!")
    rundenzahl = 1
    time.sleep(2)
    
    while held.ist_am_leben() and gegner.ist_am_leben():
        print(f"\nRunde {rundenzahl}")

        kaempfer = [held, gegner]
        random.shuffle(kaempfer)
        
        for k in kaempfer:
            if k == held:
                held.angreifen(gegner)
                if gegner.leben <= 0:
                    print(f"{gegner.name} wurde besiegt!")  
                    break
            else:
                gegner.angreifen(held)
                if held.leben <= 0:
                    print(f"{held.name} wurde besiegt!")
                    break
        time.sleep(1)
        
        
        if held.leben < 0:
            held.leben = 0
        if gegner.leben < 0:
            gegner.leben = 0
        print(f">> {held.name} hat {held.leben} Leben übrig.")
        print(f">> {gegner.name} hat {gegner.leben} Leben übrig.")
        rundenzahl += 1



    sieger = held.name if held.ist_am_leben() else gegner.name
    print(f"\nDer Sieger ist: {sieger}!")

kämpfer_klassen = [Krieger, Bogenschütze, Magier, Heiler]
gegner_klassen = [Ork, Goblin, Drache]
def spiel_starten():
    print("Wähle deinen Kämpfer:")
    for i, klasse in enumerate(kämpfer_klassen):
        print(f"{i + 1}. {klasse.__name__}")
    
    while True:
        try:
            wahl = int(input("Gib die Zahl deines Kämpfers ein: "))
            if 1 <= wahl <= len(kämpfer_klassen):
                held = kämpfer_klassen[wahl - 1](input("Gib den Namen deines Kämpfers ein: "))
                break
            else:
                print("Ungültige Auswahl. Bitte versuche es erneut.")
        except ValueError:
            print("Ungültige Eingabe. Bitte gib eine Zahl ein.")

    gegner = random.choice(gegner_klassen)()
    arena(held, gegner)

if __name__ == "__main__":
    spiel_starten() 



Wähle deinen Kämpfer:
1. Krieger
2. Bogenschütze
3. Magier
4. Heiler
Willkommen in der Arena, h!
Dein Gegner ist Drache!

Runde 1
Drache speit Feuer!
Drache trifft h mit einem Feuerstoß für 21 Schaden! (Roh: 22, Rüstung: 1)
h greift Drache an!
h trifft Drache für 6 Schaden! (Roh: 8, Rüstung: 2)
>> h hat 74 Leben übrig.
>> Drache hat 94 Leben übrig.

Runde 2
h greift Drache an!
h trifft Drache für 7 Schaden! (Roh: 9, Rüstung: 2)
Drache greift h an!
Drache trifft h für 16 Schaden! (Roh: 17, Rüstung: 1)
>> h hat 58 Leben übrig.
>> Drache hat 87 Leben übrig.

Runde 3
h greift Drache an!
h trifft Drache für 7 Schaden! (Roh: 9, Rüstung: 2)
Drache speit Feuer!
Drache trifft h mit einem Feuerstoß für 19 Schaden! (Roh: 20, Rüstung: 1)
>> h hat 39 Leben übrig.
>> Drache hat 80 Leben übrig.

Runde 4
Drache speit Feuer!
Drache trifft h mit einem Feuerstoß für 19 Schaden! (Roh: 20, Rüstung: 1)
h heilt sich selbst um 20 Leben! und hat noch 15 Heilungskraft übrig.
h greift Drache an!
Drache weicht de